### Why I simulated synthetic data?
Since Google Maps Takeout data could not be used due to access limitations and privacy constraints, I decided to generate a synthetic dataset simulating 23 years of my travel history. The goal is to build a Personal Environmental Impact Tracker that analyses my historical travel patterns and associated carbon footprint. The synthetic data mimicslife stages — from early childhood to education and work — and accounts for age-specific destinations and transport habits. 

In [2]:
import pandas as pd
import random
from datetime import datetime, timedelta

# configuration dictionary storing all simulation settings
CONFIG = {
    # start and end date of the simulation (my age)
    "date_range": {
        "start": "2001-08-28",
        "end": "2024-06-30"
    },
    # likely destinations during different life stages grouped by age brackets
    "local_locations": {
        "0-5": ["Home", "Grandparents' House", "Shopping Centre", "Park", "Cinema", "Hospital"],
        "6-17": ["Home", "School", "Park", "Shopping Centre", "Friend's Place", "Cinema", "Grandparents' House", "Cricket Ground"],
        "18-21": ["Home", "University", "Library", "Gym", "Cafe", "Friend's Place", "Cricket Ground", "Grandparents' House"],
        "22-23": ["Home", "Beach", "University", "Work", "Library", "Gym", "Cafe", "Friend's Place", "Part-time Work", "Cricket Ground", "Museum", "Hospital", "Shopping Centre", "Pharmacy"]
    },
    # transport modes with avg speed (km/h) and est. carbon emissions (g/km), gonna use these to calculate trip duration and env impact
    "transport_modes": {
        "Walk":    {"speed": 5, "co2": 0},
        "Bicycle": {"speed": 15, "co2": 0},
        "Car":     {"speed": 40, "co2": 192},
        "Bus":     {"speed": 25, "co2": 105},
        "Train":   {"speed": 45, "co2": 41},
        "Tram":    {"speed": 20, "co2": 60}
    },
    # hypothetical international travel destinations for flight emissions
    "flight_options": [
        {"destination": "Singapore", "distance_km": 12000, "co2_kg": 1500},
        {"destination": "London",    "distance_km": 34000, "co2_kg": 4500},
        {"destination": "Tokyo",     "distance_km": 17000, "co2_kg": 2300},
        {"destination": "New York",  "distance_km": 32000, "co2_kg": 4200}
    ],
    # to randomise how many international trips to generate
    "flight_settings": {
        "min_trips": 2,
        "max_trips": 4,
        "trip_length_days": 7
    }
}

# calculate age on a specific date, gonna use this to determine behavior and location choices
def age_from_date(date):
    birth_date = datetime(2001, 6, 28)
    return (date - birth_date).days // 365 # "//" for whole years only

# determine the age group label from age
def get_age_group(age):
    if age <= 5:
        return "0-5"
    elif age <= 17:
        return "6-17"
    elif age <= 21:
        return "18-21"
    else:
        return "22-23"

# a range of daily dates based on the config
def generate_date_range(config):
    return pd.date_range(start=config["date_range"]["start"], end=config["date_range"]["end"])

# international trips
def choose_international_flights(date_range, options, settings):
    n = random.randint(settings["min_trips"], settings["max_trips"])
    selected_days = sorted(random.sample(list(date_range[:-settings["trip_length_days"]]), n))
    selected_flights = random.sample(options, n)
    return list(zip(selected_days, selected_flights))

# transport mode based on age and destination
def get_mode_by_age_group(age_group, to_loc):
    if age_group == "0-5":
        return random.choices(["Car", "Bus", "Walk"], weights=[0.7, 0.2, 0.1])[0] # in my early ages, mostly travelled in my parents car (bus and walk are other likely options)
    elif age_group == "6-17" and to_loc == "School": # travelled to and from school in bus
        return "Bus"
    else:
        return random.choice(list(CONFIG["transport_modes"].keys())) # random for all the other age brackets

# single "local" trip 
def simulate_local_trip(date, from_loc, config):
    age = age_from_date(date)
    age_group = get_age_group(age)
    locations = config["local_locations"][age_group] # age-appropriate destinations 
    to_loc = random.choice([loc for loc in locations if loc != from_loc]) # destination should not be same as the origin
    mode = get_mode_by_age_group(age_group, to_loc)

    long_range_keywords = ["Home", "Work", "Part-time", "University", "Airport", "Beach"] # estimate longer distances for these locations
    if any(k in from_loc for k in long_range_keywords) or any(k in to_loc for k in long_range_keywords):
        distance = round(random.uniform(5, 20), 2)
    else:
        distance = round(random.uniform(0.5, 8), 2)

    # calc trip times and carbon emissions
    speed = config["transport_modes"][mode]["speed"]
    duration = round((distance / speed) * 60, 1) # in minutes
    co2 = round(distance * config["transport_modes"][mode]["co2"], 1)

    return {
        "Date": date.date(),
        "From": from_loc,
        "To": to_loc,
        "Mode": mode,
        "Distance_km": distance,
        "Duration_min": duration,
        "CO2_emission_g": co2,
        "Trip_Type": "Local"
    }

# round trip int flight
def simulate_flight_trip(date, flight):
    half_dist = flight["distance_km"] / 2
    half_co2 = flight["co2_kg"] * 1000 / 2
    duration = round((half_dist / 900) * 60, 1) # avg flight speed is 900 kmph

    outbound = {
        "Date": date.date(),
        "From": "Melbourne",
        "To": flight["destination"],
        "Mode": "Flight",
        "Distance_km": half_dist,
        "Duration_min": duration,
        "CO2_emission_g": half_co2,
        "Trip_Type": "International-Out"
    }

    return_date = date + timedelta(days=CONFIG["flight_settings"]["trip_length_days"])
    return_trip = {
        "Date": return_date.date(),
        "From": flight["destination"],
        "To": "Melbourne",
        "Mode": "Flight",
        "Distance_km": half_dist,
        "Duration_min": duration,
        "CO2_emission_g": half_co2,
        "Trip_Type": "International-Return"
    }

    return [outbound, return_trip]

# complete data gen 
def generate_travel_data(config=CONFIG):
    date_range = generate_date_range(config)
    flight_schedule = choose_international_flights(date_range, config["flight_options"], config["flight_settings"])
    data = []

    flight_dates = [d for d, _ in flight_schedule]
    return_dates = [d + timedelta(days=config["flight_settings"]["trip_length_days"]) for d, _ in flight_schedule]

    for date in date_range:
        if date in flight_dates:
            _, flight = [pair for pair in flight_schedule if pair[0] == date][0]
            data.extend(simulate_flight_trip(date, flight))
            continue
        if date in return_dates:
            continue # no local trips 

        age = age_from_date(date)
        age_group = get_age_group(age)

        # probability weights for no. of trips per day for different age brackets
        weight = {
            "0-5": [0.6, 0.3, 0.1, 0],
            "6-17": [0.2, 0.5, 0.2, 0.1],
            "18-21": [0.1, 0.4, 0.3, 0.2],
            "22-23": [0.1, 0.3, 0.4, 0.2]
        }

        num_trips = random.choices([0, 1, 2, 3], weights=weight[age_group])[0]
        locations = config["local_locations"][age_group]
        last_location = "Home"

        for _ in range(num_trips):
            trip = simulate_local_trip(date, last_location, config)
            data.append(trip)
            last_location = trip["To"] # use current destination as the next origin

    return pd.DataFrame(data)

df = generate_travel_data()
df.sort_values(by="Date", inplace=True)
df.to_csv("travel_data.csv", index=False)


In [3]:
df.head()

,Date,From,To,Mode,Distance_km,Duration_min,CO2_emission_g,Trip_Type
0,2001-08-28,Home,Hospital,Car,16.63,24.9,3193.0,Local
1,2001-08-28,Hospital,Grandparents' House,Car,0.58,0.9,111.4,Local
2,2001-08-31,Home,Park,Car,14.95,22.4,2870.4,Local
3,2001-09-07,Home,Park,Walk,17.76,213.1,0.0,Local
4,2001-09-12,Home,Shopping Centre,Car,5.34,8.0,1025.3,Local


In [4]:
df.shape

(9245, 8)